# Dynamic Batching: Overcoming PCIe Bandwidth Limitations

Transferring individual requests to the GPU via PCIe incurs massive latency overhead. It takes roughly the same time to send 1 request as it does to send 64 requests.
Dynamic batching queues incoming asynchronous requests and dispatches them as a single tensor to maximize PCIe throughput.


In [ ]:
import asyncio
import random
import time

class MockRouter:
    def __init__(self):
        self.queue = []
        self.batch_size = 32
    
    async def process_batch(self):
        while True:
            await asyncio.sleep(0.05) # batching window
            if self.queue:
                batch = self.queue[:self.batch_size]
                self.queue = self.queue[self.batch_size:]
                print(f"[{time.time():.3f}] Processed batch of size {len(batch)}")

    async def route(self, req_id):
        self.queue.append(req_id)

async def main():
    router = MockRouter()
    asyncio.create_task(router.process_batch())
    
    async def fire_request(req_id):
        await asyncio.sleep(random.uniform(0, 0.2))
        await router.route(req_id)
        
    tasks = [fire_request(i) for i in range(100)]
    await asyncio.gather(*tasks)
    await asyncio.sleep(0.1) # wait for last batch

# In jupyter, you would run: await main()
# await main()
print("Simulation defined.")
